In [1]:
# ---------------------------------------------------------
# 00_Setup_Compiler.ipynb
# Automated Backend Ingestion & Tool Registration Engine
# ---------------------------------------------------------

import psycopg2
import requests
import json
import urllib3

# Disable local SSL verification warnings for development environment
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Configuration for your local MES backend
SWAGGER_JSON_URL = 'https://localhost:7204/swagger/v1/swagger.json'
DB_CONFIG = {
    "dbname": "manufacturing_ai",
    "user": "postgres",
    "password": "0987654321",
    "host": "localhost",
    "port": "5432"
}

def run_setup_compiler():
    print("--- Starting MES Backend Ingestion & Compilation ---")
    
    # 1. Fetch Swagger/OpenAPI documentation from the local backend
    try:
        response = requests.get(SWAGGER_JSON_URL, verify=False)
        response.raise_for_status()
        swagger_data = response.json()
        print(f"Successfully connected to local backend: {SWAGGER_JSON_URL}")
    except Exception as e:
        print(f"Connection Error: Could not fetch Swagger JSON. Details: {e}")
        return

    # 2. Connect to the PostgreSQL Database
    try:
        conn = psycopg2.connect(**DB_CONFIG)
        cur = conn.cursor()
    except Exception as e:
        print(f"Database Connection Error: {e}")
        return

    # SQL Queries for upserting APIs and Tools
    insert_api_query = """
        INSERT INTO public.backend_api_table (
            api_id, method, end_point, description, 
            parameters, responses, tool_name, connection_id
        )
        VALUES (%s, %s, %s, %s, %s::jsonb, %s::jsonb, %s, %s)
        ON CONFLICT (api_id) DO UPDATE 
        SET end_point = EXCLUDED.end_point,
            description = EXCLUDED.description,
            parameters = EXCLUDED.parameters,
            responses = EXCLUDED.responses;
    """

    insert_tool_query = """
        INSERT INTO public.tool_table (
            tool_name, description, tool_type, status, customer_id
        )
        VALUES (%s, %s, %s, %s, %s)
        ON CONFLICT (tool_name) DO NOTHING;
    """

    api_count = 0
    tool_count = 0
    connection_id = 1  # Linked to the MES connection record created earlier

    # 3. Parse Paths, Methods, and Operations from OpenAPI spec
    paths = swagger_data.get('paths', {})
    for path, path_item in paths.items():
        for method, operation in path_item.items():
            if method.lower() not in ['get', 'post', 'put', 'delete', 'patch']:
                continue
                
            # Generate deterministic identifiers
            clean_path_id = path.replace('/', '_').replace('{', '').replace('}', '').strip('_')
            api_id = f"api_{method.lower()}_{clean_path_id}"[:50]
            
            tool_name = operation.get('operationId', f"{method}_{clean_path_id}")
            description = operation.get('summary', operation.get('description', 'Automated MES endpoint'))
            
            parameters = json.dumps(operation.get('parameters', []))
            responses = json.dumps(operation.get('responses', {}))

            # Execute API table insert
            cur.execute(insert_api_query, (
                api_id,
                method.upper(),
                path,
                description,
                parameters,
                responses,
                tool_name,
                connection_id
            ))
            api_count += 1

            # Register as an available tool in tool_table for agent binding
            cur.execute(insert_tool_query, (
                tool_name,
                description,
                "MES_API_Adapter",
                "active",
                1
            ))
            tool_count += 1

    # Commit transactions
    conn.commit()
    cur.close()
    conn.close()
    
    print(f"Compilation Complete!")
    print(f" -> Successfully synced {api_count} API endpoints into `backend_api_table`.")
    print(f" -> Automatically registered {tool_count} operational tools into `tool_table`.")

# Execute the compiler
if __name__ == "__main__":
    run_setup_compiler()

--- Starting MES Backend Ingestion & Compilation ---
Successfully connected to local backend: https://localhost:7204/swagger/v1/swagger.json
Compilation Complete!
 -> Successfully synced 670 API endpoints into `backend_api_table`.
 -> Automatically registered 670 operational tools into `tool_table`.
